DUPLICATE DETECTION AND DATA DEDUPLICATION

In [1]:
import pandas as pd
from ydata_profiling import ProfileReport
import recordlinkage

In [2]:
# Load the database after handling outliers
db = pd.read_csv('db_outliers_handled.csv')

In [3]:
# Convert data types
bool_cols = ['Alimentare', 'Non Alimentare', 'Tabella Speciale Carburanti', 'Tabella Speciale Farmacie', 'Tabella Speciale Monopolio']
db[bool_cols] = db[bool_cols].astype("boolean")

db["Codice Via"] = db["Codice Via"].astype("Int64")
db["ZD"] = db["ZD"].astype("Int64")

In [4]:
# Detect duplicates
duplicated = db.duplicated()

# Count duplicates
num_duplicates = db.duplicated().sum()
num_duplicates

np.int64(62)

In [5]:
# Remove duplicates
db = db.drop_duplicates()

In [6]:
# Record linkage to find potential duplicates based on 'Via' column
indexer = recordlinkage.Index()
indexer.block('Via')
candidate_links = indexer.index(db)

In [7]:
# Compare the candidate pairs
compare_cl = recordlinkage.Compare()

# Columns that must match exactly
exact_cols = [
    'Alimentare', 
    'Non Alimentare', 
    'Tabella Speciale Carburanti',       
    'Tabella Speciale Farmacie', 
    'Tabella Speciale Monopolio', 
    'Tipo Via', 
    'Civico', 
    'Codice Via', 
    'ZD', 
    'Isolato', 
    'Accesso'
]

for col in exact_cols:
    compare_cl.exact(col, col, label=col)

# Fuzzy matching for 'Via' and 'Settore Storico Cf Preval' columns
compare_cl.string('Via', 'Via', method='jarowinkler', label='Via')
compare_cl.string('Settore Storico Cf Preval', 'Settore Storico Cf Preval', method='jarowinkler', threshold=0.85, label='Settore_Storico_Cf_Preval')

# Generate an array of features (0 if the condition is not satisfied and 1 if the condition is satisfied)
features = compare_cl.compute(candidate_links, db)

In [8]:
# Identify matches where all 13 features match
matches = features[features.sum(axis=1)>12]

n_distinct = matches.index.get_level_values(1).nunique()
n_distinct

620

In [9]:
# Remove identified duplicates from the original database
DATA_DROP_DUPLICATES = db.copy()
indexes = []
for i in matches.index:
    if i[1] not in indexes:
        DATA_DROP_DUPLICATES = DATA_DROP_DUPLICATES.drop([i[1]])
    indexes.append(i[1])

db = DATA_DROP_DUPLICATES

In [10]:
# Save the final database without duplicates
db.to_csv('db_final.csv')

In [11]:
# Generate the profiling report
PROFILE = ProfileReport(db, title="Profiling Report").to_file("db_REPORT_final.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 89.13it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]